In [2]:
# ---------------------------------------------------------
# Cell 1: 匯入所有必要套件
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import warnings
import random
import os
import matplotlib.pyplot as plt
import optuna

from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, StackingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# 補上評估指標套件 (手動防漏 CV 需要)
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, log_loss

random.seed(42)
np.random.seed(42)
warnings.filterwarnings('ignore')

In [3]:
# ---------------------------------------------------------
# Cell 2: 基礎資料清洗、極端值防禦與 Target Encoding
# ---------------------------------------------------------
# 1. 讀取原始資料
train_df = pd.read_csv('../data/boy or girl 2025 train_missingValue.csv')
test_df = pd.read_csv('../data/boy or girl 2025 test no ans_missingValue.csv')

# 2. 清理 phone_os 噪音值
def clean_os(val):
    val = str(val).lower()
    if 'apple' in val or 'ios' in val: return 'Apple'
    if 'android' in val: return 'Android'
    return 'Other'

train_df['phone_os'] = train_df['phone_os'].apply(clean_os)
test_df['phone_os'] = test_df['phone_os'].apply(clean_os)

# 🚀 修正：計算 99 百分位數作為 yt 合理上限
train_df['yt'] = pd.to_numeric(train_df['yt'], errors='coerce')
yt_99_percentile = train_df['yt'].dropna().quantile(0.99)
print(f"yt 欄位 99% 上限設定為: {yt_99_percentile:.0f}")

# 3. 物理邊界強化 (嚴格過濾極端值，防溢位)
def apply_advanced_bounds(df):
    df_clean = df.copy()
    df_clean['yt'] = pd.to_numeric(df_clean['yt'], errors='coerce')
    df_clean['height'] = df_clean['height'].apply(lambda x: x if 100 <= x <= 250 else np.nan)
    df_clean['weight'] = df_clean['weight'].apply(lambda x: x if 30 <= x <= 200 else np.nan)
    df_clean['iq'] = df_clean['iq'].apply(lambda x: x if 50 <= x <= 200 else np.nan)
    df_clean['sleepiness'] = df_clean['sleepiness'].apply(lambda x: x if 0 <= x <= 24 else np.nan)
    df_clean['fb_friends'] = df_clean['fb_friends'].apply(lambda x: x if 0 <= x <= 5000 else np.nan)
    # 🚀 修正：使用 99 百分位數取代原本寬鬆的 1e15
    df_clean['yt'] = df_clean['yt'].apply(lambda x: x if 0 <= x <= yt_99_percentile else np.nan)
    return df_clean

train_df = apply_advanced_bounds(train_df)
test_df = apply_advanced_bounds(test_df)

# 4. 保留類別特徵
X = train_df.drop(columns=['id', 'gender', 'self_intro'])
y = train_df['gender']
X_test_submission = test_df.drop(columns=['id', 'gender', 'self_intro'])
test_ids = test_df['id']

le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

# 🚨 修正：為防止 Target Encoding 洩漏，原本的 fit_transform 已移至 Cell 4 的防漏函數內
# print("執行 Target Encoding (Smoothing=20)...")
# te = TargetEncoder(cols=['star_sign', 'phone_os'], smoothing=20)
# X = te.fit_transform(X, y_encoded)
# X_test_submission = te.transform(X_test_submission)
print("📌 Target Encoding 已延後至 CV 內部執行以防止 Data Leakage。")

# 5. 建立缺失值指示器 (保留重要的缺失資訊)
for col in ['weight', 'height', 'yt']:
    X[f'{col}_is_missing'] = X[col].isnull().astype(int)
    X_test_submission[f'{col}_is_missing'] = X_test_submission[col].isnull().astype(int)

print(f"✅ Cell 2 處理完成！目前特徵總數: {X.shape[1]}")

yt 欄位 99% 上限設定為: 99999
📌 Target Encoding 已延後至 CV 內部執行以防止 Data Leakage。
✅ Cell 2 處理完成！目前特徵總數: 11


In [4]:
# ---------------------------------------------------------
# Cell 3: 升級版 NLP 語意萃取 (BGE-Large) 與 PCA 降維
# ---------------------------------------------------------
print("載入 BGE-Large 模型擷取文字語意 (首次執行需下載模型)...")

# 🚀 修正：將 Prefix 設為空字串，避免扭曲分類任務的向量空間
prefix = ""
def process_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return 'no description'
    return prefix + str(text)

train_texts = [process_text(t) for t in train_df['self_intro']]
test_texts = [process_text(t) for t in test_df['self_intro']]

# 🚀 升級武器 1：使用 BGE-Large 擷取深層語意
embedder = SentenceTransformer('BAAI/bge-large-en-v1.5')
train_emb = embedder.encode(train_texts, show_progress_bar=True)
test_emb = embedder.encode(test_texts, show_progress_bar=True)

print("📌 PCA 降維已延後至 CV 內部執行以防止 Data Leakage。")
print("✅ NLP 處理完成！(無 Prefix 純淨版)")

載入 BGE-Large 模型擷取文字語意 (首次執行需下載模型)...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Batches:   0%|          | 0/14 [00:00<?, ?it/s]


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# ---------------------------------------------------------
# Cell 4: 隨機森林進階補值與特徵工程
# ---------------------------------------------------------
print("建立進階衍生特徵函數 ( BMI, 體型比例, Log Transform )...")

# 📌 完美保留你的特徵工程邏輯
def create_advanced_features(df):
    df_new = df.copy()
    
    # 1. 建立 BMI 特徵
    df_new['height'] = df_new['height'].clip(lower=1) 
    df_new['BMI'] = df_new['weight'] / ((df_new['height'] / 100) ** 2)
    
    # 2. 加入線性的身高體重比 (來自 Willy)
    df_new['height_weight_ratio'] = df_new['height'] / df_new['weight']
    
    # 3. 對極端右偏的數值進行 Log1p 轉換
    df_new['yt_log'] = np.log1p(df_new['yt'].clip(lower=0))
    df_new['fb_friends_log'] = np.log1p(df_new['fb_friends'].clip(lower=0))
    
    # 移除原始尚未 Log 的欄位
    df_new = df_new.drop(columns=['yt', 'fb_friends'])
    return df_new

def preprocess_fold(X_tr, y_tr, X_va, tr_emb, va_emb):
    X_tr, X_va = X_tr.copy(), X_va.copy()
    
    # (a) Target Encoding
    te = TargetEncoder(cols=['star_sign', 'phone_os'], smoothing=20)
    X_tr = te.fit_transform(X_tr, y_tr)
    X_va = te.transform(X_va)
    
    # 🚀 修正：(b) PCA 維度提升至 5 維 (迴圈同步改為 5)
    pca = PCA(n_components=5, random_state=42)
    tr_pca = pca.fit_transform(tr_emb)
    va_pca = pca.transform(va_emb)
    for i in range(5):
        X_tr[f'intro_pca_{i}'] = tr_pca[:, i]
        X_va[f'intro_pca_{i}'] = va_pca[:, i]
        
    # (c) Random Forest Imputer
    rf_imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), 
        random_state=42, max_iter=10
    )
    X_tr_imp = pd.DataFrame(rf_imputer.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_va_imp = pd.DataFrame(rf_imputer.transform(X_va), columns=X_va.columns, index=X_va.index)
    
    # (d) 執行特徵工程
    return create_advanced_features(X_tr_imp), create_advanced_features(X_va_imp)

print("✅ 預處理防漏生產線準備完成！(確保 Imputer 只看 Training Set)")

建立進階衍生特徵函數 ( BMI, 體型比例, Log Transform )...
✅ 預處理防漏生產線準備完成！(確保 Imputer 只看 Training Set)


In [ ]:
# ---------------------------------------------------------
# Cell 5: 科學化特徵淘汰 (多次投票制 Permutation Importance)
# ---------------------------------------------------------
print("🚀 升級武器 3：執行多次投票制 Permutation Importance 自動抓出噪聲特徵...")

# 使用 LightGBM 作為 PI 測試模型
lgbm_pi = LGBMClassifier(class_weight='balanced', random_state=42, max_depth=4, verbose=-1)

# 大師級建議：使用 5 個不同的亂數種子進行投票
seeds = [42, 123, 456, 789, 1024]
all_keep_masks = []
features = None

print(f"開始計算 {len(seeds)} 輪的特徵重要性，請稍候...")

for seed in seeds:
    cv_pi = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    perm_scores_this_round = []
    
    for tr_idx, val_idx in cv_pi.split(X, y_encoded):
        X_tr, y_tr = X.iloc[tr_idx], y_encoded[tr_idx]
        X_val, y_val = X.iloc[val_idx], y_encoded[val_idx]
        tr_emb, val_emb = train_emb[tr_idx], train_emb[val_idx]
        
        # 嚴格防漏預處理
        X_tr_eng, X_val_eng = preprocess_fold(X_tr, y_tr, X_val, tr_emb, val_emb)
        if features is None: features = X_tr_eng.columns
        
        lgbm_pi.fit(X_tr_eng, y_tr)
        
        # 計算這一個 Fold 的特徵重要性
        result = permutation_importance(
            lgbm_pi, X_val_eng, y_val,
            n_repeats=10, random_state=seed, scoring='roc_auc'
        )
        perm_scores_this_round.append(result.importances_mean)
        
    # 算出這一個 Seed (5-Fold 平均) 的分數
    round_mean = np.mean(perm_scores_this_round, axis=0)
    all_keep_masks.append(round_mean > 0)

# 🚀 只要有任何一輪認為有用，就保留 (np.any)
final_keep_mask = np.sum(all_keep_masks, axis=0) >= 3

removed_features = [features[i] for i in range(len(features)) if not final_keep_mask[i]]
kept_features = [features[i] for i in range(len(features)) if final_keep_mask[i]]

print(f"\n🚫 成功移除「每一輪都被判定無用」的噪聲特徵: {removed_features}")
print(f"✅ 保留核心特徵 ({len(kept_features)} 個): {kept_features}")

🚀 升級武器 3：執行多次投票制 Permutation Importance 自動抓出噪聲特徵...
開始計算 5 輪的特徵重要性，請稍候...


c:\Users\User\Desktop\boys_or_girls\venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_sile


🚫 成功移除「每一輪都被判定無用」的噪聲特徵: ['intro_pca_3', 'fb_friends_log']
✅ 保留核心特徵 (16 個): ['star_sign', 'phone_os', 'height', 'weight', 'sleepiness', 'iq', 'weight_is_missing', 'height_is_missing', 'yt_is_missing', 'intro_pca_0', 'intro_pca_1', 'intro_pca_2', 'intro_pca_4', 'BMI', 'height_weight_ratio', 'yt_log']


In [ ]:
# ---------------------------------------------------------
# Cell 6: 三檔位融合 (Blending) 驗證與綜合評估
# ---------------------------------------------------------
print("啟動大師級融合策略：手動三檔位 Blending (目標: 中和過擬合，極大化穩定性)...")

ratio = (y_encoded == 0).sum() / (y_encoded == 1).sum()

# 🚀 定義三組不同「性格」的參數
configs = {
    'Conservative (保守)': {
        'xgb': {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 150, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 5.0},
        'lgbm': {'max_depth': 3, 'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 150},
        'rf': {'max_depth': 4, 'n_estimators': 150, 'min_samples_leaf': 4}
    },
    'Moderate (中庸)': {
        'xgb': {'max_depth': 5, 'learning_rate': 0.03, 'n_estimators': 200, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0},
        'lgbm': {'max_depth': 5, 'num_leaves': 31, 'learning_rate': 0.03, 'n_estimators': 200},
        'rf': {'max_depth': 6, 'n_estimators': 200, 'min_samples_leaf': 2}
    },
    'Aggressive (激進)': {
        'xgb': {'max_depth': 7, 'learning_rate': 0.01, 'n_estimators': 300, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 0.1},
        'lgbm': {'max_depth': 7, 'num_leaves': 63, 'learning_rate': 0.01, 'n_estimators': 300},
        'rf': {'max_depth': 8, 'n_estimators': 300, 'min_samples_leaf': 1}
    }
}

# ==========================================
# 預先計算 10-Fold 防漏資料 (加速神技)
# ==========================================
print("⏳ 預先計算 10-Fold 防漏資料 (只算一次，確保完全零洩漏)...")
skf_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
precomputed_10_folds = []

for tr_idx, val_idx in skf_10.split(X, y_encoded):
    X_tr, y_tr = X.iloc[tr_idx], y_encoded[tr_idx]
    X_va, y_va = X.iloc[val_idx], y_encoded[val_idx]
    tr_emb, va_emb = train_emb[tr_idx], train_emb[val_idx]
    
    X_tr_eng, X_va_eng = preprocess_fold(X_tr, y_tr, X_va, tr_emb, va_emb)
    X_tr_pruned, X_va_pruned = X_tr_eng[kept_features], X_va_eng[kept_features]
    precomputed_10_folds.append((X_tr_pruned, y_tr, X_va_pruned, y_va))

print("✅ 10-Fold 資料準備完成！開始執行 Blending 融合評估...\n")

acc_list, auc_list, f1_list, logloss_list = [], [], [], []

# 在每一個 Fold 裡面，同時訓練三個模型並平均它們的機率
fold_count = 1
for X_tr_p, y_tr_p, X_va_p, y_va_p in precomputed_10_folds:
    print(f"訓練 Fold {fold_count}/10 ...")
    fold_probs = []
    
    for name, cfg in configs.items():
        xgb_clf = XGBClassifier(scale_pos_weight=ratio, eval_metric='logloss', random_state=42, tree_method='hist', device='cuda', **cfg['xgb'])
        lgbm_clf = LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1, **cfg['lgbm'])
        rf_clf = RandomForestClassifier(class_weight='balanced', random_state=42, **cfg['rf'])
        
        model = StackingClassifier(
            estimators=[('XGB', xgb_clf), ('LGBM', lgbm_clf), ('RF', rf_clf)],
            final_estimator=LogisticRegression(class_weight='balanced', random_state=42)
        )
        
        model.fit(X_tr_p, y_tr_p)
        fold_probs.append(model.predict_proba(X_va_p))
    
    # 🚀 核心融合魔法：將三個模型的預測機率直接相加除以 3
    blended_prob = np.mean(fold_probs, axis=0)
    blended_class = np.argmax(blended_prob, axis=1)  # 機率大的那一邊獲勝
    
    acc_list.append(accuracy_score(y_va_p, blended_class))
    auc_list.append(roc_auc_score(y_va_p, blended_prob[:, 1]))
    f1_list.append(f1_score(y_va_p, blended_class))
    logloss_list.append(log_loss(y_va_p, blended_prob))
    
    fold_count += 1

print("\n🏆 三神融合 (Blending) 評估完成！")

print("\n📊 === Ultimate 終極融合版 Blending 10-Fold CV 綜合評估報告 ===")
print(f"Accuracy (準確率) : {np.mean(acc_list):.4f} (std: {np.std(acc_list):.4f})")
print(f"ROC-AUC  (排序力) : {np.mean(auc_list):.4f} (std: {np.std(auc_list):.4f})")
print(f"F1-Score (平衡度) : {np.mean(f1_list):.4f} (std: {np.std(f1_list):.4f})")
print(f"Log Loss (誤差值) : {np.mean(logloss_list):.4f} (std: {np.std(logloss_list):.4f})")

啟動大師級融合策略：手動三檔位 Blending (目標: 中和過擬合，極大化穩定性)...
⏳ 預先計算 10-Fold 防漏資料 (只算一次，確保完全零洩漏)...


C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refr

✅ 10-Fold 資料準備完成！開始執行 Blending 融合評估...

訓練 Fold 1/10 ...
訓練 Fold 2/10 ...
訓練 Fold 3/10 ...
訓練 Fold 4/10 ...
訓練 Fold 5/10 ...
訓練 Fold 6/10 ...
訓練 Fold 7/10 ...
訓練 Fold 8/10 ...
訓練 Fold 9/10 ...
訓練 Fold 10/10 ...

🏆 三神融合 (Blending) 評估完成！

📊 === Ultimate 終極融合版 Blending 10-Fold CV 綜合評估報告 ===
Accuracy (準確率) : 0.8792 (std: 0.0559)
ROC-AUC  (排序力) : 0.9215 (std: 0.0750)
F1-Score (平衡度) : 0.7841 (std: 0.0908)
Log Loss (誤差值) : 0.3868 (std: 0.1163)


In [ ]:
# ---------------------------------------------------------
# Cell 7: 終極融合版 (Blending) 預測與存檔
# ---------------------------------------------------------
print("正在將全局資料通過防漏生產線進行轉換...")
X_train_final, X_test_final = preprocess_fold(X, y_encoded, X_test_submission, train_emb, test_emb)

X_pruned = X_train_final[kept_features]
X_test_pruned = X_test_final[kept_features]

print(f"目前訓練集特徵數: {X_pruned.shape[1]}") 
print(f"目前測試集特徵數: {X_test_pruned.shape[1]}") 

print("\n正式訓練三組 Ultimate Stacking 模型，並進行預測融合中...")
test_fold_probs = []

for name, cfg in configs.items():
    print(f"  -> 訓練並預測 {name} 模型...")
    xgb_clf = XGBClassifier(scale_pos_weight=ratio, eval_metric='logloss', random_state=42, tree_method='hist', device='cuda', **cfg['xgb'])
    lgbm_clf = LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1, **cfg['lgbm'])
    rf_clf = RandomForestClassifier(class_weight='balanced', random_state=42, **cfg['rf'])
    
    model = StackingClassifier(
        estimators=[('XGB', xgb_clf), ('LGBM', lgbm_clf), ('RF', rf_clf)],
        final_estimator=LogisticRegression(class_weight='balanced', random_state=42)
    )
    
    model.fit(X_pruned, y_encoded)
    test_fold_probs.append(model.predict_proba(X_test_pruned))

print("\n執行最終機率平均融合 (Blending)...")
# 🚀 融合三組預測機率
final_blended_prob = np.mean(test_fold_probs, axis=0)
test_predictions_encoded = np.argmax(final_blended_prob, axis=1)

# 將 0, 1 轉回原始的性別標籤
test_predictions_original = le_y.inverse_transform(test_predictions_encoded)

submission_df = pd.DataFrame({
    'id': test_ids, 
    'gender': test_predictions_original
})

output_dir = '../results'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, 'submission_blending_ultimate.csv')
submission_df.to_csv(file_path, index=False)

print(f"🏆 終極融合版預測成功！檔案已儲存至: {file_path}")
print("前五筆預測結果：")
print(submission_df.head())

正在將全局資料通過防漏生產線進行轉換...


C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python312\Lib\contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
C:\Python312\Lib\contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refr

目前訓練集特徵數: 16
目前測試集特徵數: 16

正式訓練三組 Ultimate Stacking 模型，並進行預測融合中...
  -> 訓練並預測 Conservative (保守) 模型...
  -> 訓練並預測 Moderate (中庸) 模型...
  -> 訓練並預測 Aggressive (激進) 模型...

執行最終機率平均融合 (Blending)...
🏆 終極融合版預測成功！檔案已儲存至: ../results\submission_blending_ultimate.csv
前五筆預測結果：
   id  gender
0   1       1
1   2       1
2   3       2
3   4       1
4   5       2
